# Presencia de mujeres en la dirección de películas (2000-2025)

**Proyecto de análisis exploratorio de datos (EDA)** · María Balagué García · The Bridge, Data Science

Esta memoria resume todo el proceso: de dónde salen los datos, cómo los
limpié, qué analicé y qué conclusiones se pueden sacar. El código completo
está en `src/notebooks/`; aquí solo se muestran los pasos esenciales y se
leen los resultados ya calculados (`src/data/resultados/`), de modo que
la memoria se puede ejecutar sin volver a descargar los datos.

**Índice:** 1. Introducción · 2. Fuentes de datos · 3. Limpiezas ·
4. Desarrollo del análisis · 5. Medidas estadísticas · 6. Visualizaciones ·
7. Conclusiones

## 1. Introducción

### Pregunta de estudio

¿Qué géneros cinematográficos dirigen más las mujeres, en comparación con
los hombres, y cómo ha cambiado su presencia entre 2000 y 2025?

Los datos son todas las películas estrenadas entre 2000 y 2025 que figuran
en TMDB (The Movie Database), una base de datos abierta mantenida por su
comunidad.

### Hipótesis iniciales (formuladas antes de analizar los datos)

1. En todos los géneros hay más hombres que mujeres en la dirección.
2. La presencia de mujeres en la dirección ha aumentado en los últimos 25 años.
3. El drama es el género que más dirigen las mujeres.

### Hipótesis revisada (formulada después de explorar los datos)

Tras la exploración añadí una cuarta, más concreta: *en todos los géneros
con suficientes películas, las mujeres y personas no binarias dirigen menos
del 50 %*. La presento como revisada y no como una predicción previa,
porque la formulé al ver que el porcentaje global rondaba el 20 %.

### Sobre las valoraciones

El enunciado inicial incluía comparar valoraciones por grupo de dirección.
Lo exploré, pero lo descarté: el 60,79 % de las películas (414.488) no tiene
ningún voto, así que su valoración no existe. En su lugar uso el número de votos como
**nivel de visibilidad** (todas las películas, al menos 1 voto, al menos
100 votos) para comprobar si los resultados cambian entre películas poco y
muy vistas.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))          # para importar la carpeta utils
from utils.direccion import (buscar_raiz, leer_resultado, clasificar_direccion,
                             intervalo_confianza_95, formato_miles)

raiz = buscar_raiz()
pd.set_option("display.width", 160)
print("Rutas listas")

Rutas listas


## 2. Fuentes de datos

**Origen.** API de TMDB, con dos consultas:

- `/discover/movie`: listado de películas por fechas de estreno. La API
  limita cada búsqueda a 500 páginas, así que dividí el periodo en 312
  intervalos mensuales. Se descargaron 34.243 páginas con **681.868
  películas** (título, géneros, fecha, votos, idioma, etc.).
- `/movie/{id}/credits`: reparto y equipo de cada película, de donde sale
  quién dirige y el código de género de esa persona. Se obtuvieron los
  créditos de 681.836 películas; 32 devolvieron un error 404 (0,005 %) y se
  documentaron como «sin créditos».

**Cómo se descargó.** Pedir 681.868 créditos uno a uno habría llevado días,
así que usé 4 hilos de ejecución en paralelo (la espera de la red es lo
que más tiempo consume). La descarga se podía reanudar (si un archivo ya
existe no se vuelve a pedir) y reintentaba automáticamente los errores
temporales. Tardó 487,8 minutos.

**Problemas encontrados.**

- El tope de 500 páginas obligó a trabajar por meses.
- El total anual que informa la API no coincide con la suma de sus meses, y
  varía de una ejecución a otra (TMDB cambia mientras se descarga).
- Los errores 429 y 5xx se resolvieron con reintentos; los 404 no tienen
  solución y se registraron.
- El token de acceso se guarda en un archivo `.env` fuera del repositorio.

Notebooks: `00_pruebas_api`, `01_obtencion_datos`, `01b_revision_errores`.

## 3. Limpiezas

Notebooks: `02_limpieza_datos`, `02b_direccion`.

**Fechas.** El 27,4 % de las películas de 2000 tiene como fecha el 1 de
enero, lo que sugiere una fecha de relleno cuando se desconoce el día real
(en 2025 es el 2,3 %). Por eso analizo solo por **año**, nunca por mes.

**Géneros.** Una película puede tener varios géneros. El 22,6 % (154.099)
no tiene ninguno informado; las conservo con la etiqueta «Sin género» y las
muestro siempre aparte de los géneros reales.

**Valoraciones.** Se descartó su uso (ver introducción). Las películas sin
votos se conservan para el análisis de géneros y de evolución temporal.

**Dirección.** Del equipo de cada película me quedo con las personas cuyo
puesto es exactamente `Director` (690.938 créditos de dirección). TMDB
informa un código de género por persona: 0 sin especificar, 1 mujer,
2 hombre, 3 no binaria. Con esos códigos clasifico cada película en un grupo:

| Grupo | Regla |
|---|---|
| Mujeres (y personas no binarias) | todas las personas con código 1 o 3 |
| Hombres | todas con código 2 |
| Mixto | hay personas de ambos grupos |
| No identificado | alguna persona con código 0 |
| Sin dirección acreditada | hay créditos pero ningún `Director` (descartada) |
| Sin créditos descargados | error 404 (descartada) |

Si una película tiene una persona sin género especificado, va a «No
identificado» aunque la otra sea una mujer, porque no se puede saber cómo es
el equipo. Esa regla afecta a un 15,3 % de ese grupo; las personas no
binarias son como máximo el 4,3 % del grupo «Mujeres». La regla está
implementada en `src/utils/direccion.py`:

In [2]:
# Ejemplos de cómo clasifica una película la función del proyecto
ejemplos = {"Una directora": [1], "Un director": [2], "Directora y director": [1, 2],
            "Directora y una persona sin género en TMDB": [1, 0], "Dos personas no binarias o mujeres": [3, 1]}
for descripcion, codigos in ejemplos.items():
    print(f"{descripcion:45} {codigos!s:8} -> {clasificar_direccion(codigos)}")

Una directora                                 [1]      -> Mujeres
Un director                                   [2]      -> Hombres
Directora y director                          [1, 2]   -> Mixto
Directora y una persona sin género en TMDB    [1, 0]   -> No identificado
Dos personas no binarias o mujeres            [3, 1]   -> Mujeres


**Resultado de la clasificación.** De las 681.868 películas, 602.451
tienen dirección acreditada y forman la base del análisis:

In [3]:
grupos = leer_resultado("grupos_direccion.csv").set_index("grupo_direccion")
grupos["%"] = (grupos["n"] / grupos["n"].sum() * 100).round(1)
grupos["n"] = grupos["n"].map(formato_miles)
display(grupos)

,n,%
grupo_direccion,,
No identificado,322.770,47.3
Hombres,219.028,32.1
Sin dirección acreditada,79.385,11.6
Mujeres,56.466,8.3
Mixto,4.187,0.6
Sin créditos descargados,32,0.0


**Dos cautelas importantes de esta limpieza:**

- **«No identificado» es casi la mitad** del total (47,3 %), y el 53,6 % de
  la base del análisis. Por eso todos los porcentajes se calculan sobre las
  películas de **dirección conocida** (Mujeres, Hombres y Mixto: 279.681).
- **El descarte no es aleatorio.** Se descartó el 19,5 % de las películas
  de 2005 pero solo el 7,4 % de las de 2025, y las descartadas tienen menos
  votos (22,0 % con votos frente a 41,5 % en las conservadas).

## 4. Desarrollo del análisis

Notebooks: `03a_analisis_visualizacion`, `03b_comparacion_direccion`.

1. **Descripción del conjunto completo** (sin tener en cuenta la dirección):
   películas por año, géneros más frecuentes y su evolución.
2. **Comparación por grupo de dirección**: porcentaje de películas dirigidas
   por mujeres y personas no binarias en cada género, con tres niveles de
   visibilidad (todas las películas, ≥ 1 voto, ≥ 100 votos). Con ≥ 100 votos
   exijo al menos 100 películas por género; en los otros dos, 500.
3. **Evolución temporal**: por año y por periodos de cinco años.

Como el catálogo de TMDB crece mucho con los años, comparo **proporciones y
no recuentos**.

## 5. Medidas estadísticas

- **Porcentaje por género**: películas dirigidas por mujeres y personas no
  binarias entre las de dirección conocida de ese género.
- **Intervalo de confianza al 95 %** de cada porcentaje (aproximación normal):
  $p \pm 1{,}96\sqrt{p(1-p)/n}$. Son las barras de error de las gráficas.
- **Pendiente de la tendencia**: recta de mínimos cuadrados del porcentaje
  anual (puntos porcentuales por año).
- **Comparación de bloques**: 2000-2004 frente a 2021-2025.
- **Mujeres por cada 100 películas de hombres**: cociente entre las películas
  dirigidas por mujeres y personas no binarias y las dirigidas por hombres
  (sin las mixtas). Sirve para comprobar que el aumento no se debe solo a que
  haya más películas.

In [4]:
evol = leer_resultado("evolucion_mujeres.csv")

filas = []
for nivel, g in evol.groupby("nivel", sort=False):
    pendiente = np.polyfit(g["anio_estreno"], g["pct"], 1)[0]
    ini = g[g["anio_estreno"].between(2000, 2004)]
    fin = g[g["anio_estreno"].between(2021, 2025)]
    filas.append({"nivel": nivel,
                  "pendiente (pp/año)": round(pendiente, 2),
                  "2000-04 (%)": round(ini["n_mujeres"].sum() / ini["n"].sum() * 100, 1),
                  "2021-25 (%)": round(fin["n_mujeres"].sum() / fin["n"].sum() * 100, 1)})
display(pd.DataFrame(filas).set_index("nivel"))

,pendiente (pp/año),2000-04 (%),2021-25 (%)
nivel,,,
Todas las películas,0.54,13.5,24.2
Con al menos 1 voto,0.48,12.0,21.3
Con al menos 100 votos,0.48,6.4,16.2


El porcentaje de películas dirigidas por mujeres y personas no binarias
sube en los tres niveles, entre 0,47 y 0,54 puntos porcentuales por año.

## 6. Visualizaciones

### 6.1 El catálogo: cuántas películas y de qué géneros

![Películas por año](../figuras/01_peliculas_por_anio.png)

El catálogo pasa de unas 8.700 películas en 2000 a más de 55.000 en 2025.
Gran parte de ese crecimiento es que TMDB cataloga más, no solo que se
produzca más cine; por eso el análisis usa proporciones.

![Géneros más frecuentes](../figuras/02_generos_frecuentes.png)

Drama (25,0 %) y documental (23,5 %) son los géneros más frecuentes. El
22,6 % de las películas no tiene género informado (en gris).

![Evolución de los géneros](../figuras/03_evolucion_generos.png)

Terror y suspense ganan peso desde 2020; comedia cae en 2020-2021 y se
recupera después.

### 6.2 Quién dirige: el grupo de dirección

![Películas según el grupo de dirección](../figuras/05_grupos_direccion.png)

Solo el 41 % de las películas tiene una dirección con género conocido. Las
barras grises (no identificado, sin dirección acreditada, sin créditos) no
entran en las comparaciones.

In [5]:
# Porcentaje de dirección conocida dirigida por mujeres y personas no binarias
# (Mujeres + Hombres + Mixto = 279.681 películas)
g = leer_resultado("grupos_direccion.csv").set_index("grupo_direccion")["n"]
conocida = g[["Hombres", "Mujeres", "Mixto"]]
total = conocida.sum()
print(formato_miles(total), "películas de dirección conocida")
for k, v in conocida.items():
    print(f"  {k:8} {v / total * 100:5.1f} %")

279.681 películas de dirección conocida
  Hombres   78.3 %
  Mujeres   20.2 %
  Mixto      1.5 %


### 6.3 ¿En qué géneros dirigen más las mujeres?

![Mujeres por género, todas las películas](../figuras/06a_mujeres_por_genero_todas.png)

Con todas las películas, el documental (26,7 %) y la animación (25,7 %)
son los géneros con más presencia; acción (3,7 %) es el que menos. La línea
roja marca el 50 %: ningún género se acerca. Las barras de error son los
intervalos al 95 %.

![Mujeres por género, al menos 1 voto](../figuras/06b_mujeres_por_genero_1voto.png)

![Mujeres por género, al menos 100 votos](../figuras/06c_mujeres_por_genero_100votos.png)

Al exigir votos, los porcentajes bajan y el orden cambia: con al menos 100
votos el máximo es romance (19,0 %) y acción sigue siendo el mínimo (2,8 %).
Con al menos 100 votos hay menos películas y los intervalos son más anchos
(documental: 17,8 %, con un intervalo de ±3,2 puntos).

In [6]:
niveles = {"Todas las películas": "presencia_genero_todas.csv",
           "Al menos 1 voto": "presencia_genero_1voto.csv",
           "Al menos 100 votos": "presencia_genero_100votos.csv"}
filas = []
for nombre, archivo in niveles.items():
    t = leer_resultado(archivo)
    t = t[t["genero"] != "Sin género"]            # «Sin género» no es un género real
    mx, mn = t.loc[t["pct_mujeres"].idxmax()], t.loc[t["pct_mujeres"].idxmin()]
    filas.append({"nivel": nombre,
                  "máximo": f"{mx['genero']} ({mx['pct_mujeres']} %)",
                  "mínimo": f"{mn['genero']} ({mn['pct_mujeres']} %)"})
display(pd.DataFrame(filas).set_index("nivel"))

,máximo,mínimo
nivel,,
Todas las películas,Documental (26.72 %),Acción (3.74 %)
Al menos 1 voto,Documental (23.21 %),Acción (3.43 %)
Al menos 100 votos,Romance (18.98 %),Acción (2.82 %)


### 6.4 Qué géneros dirigen unas y otros

![Géneros dentro de cada grupo](../figuras/07_generos_por_grupo.png)

Esta gráfica cambia la pregunta: dentro de la filmografía de cada grupo,
¿qué peso tiene cada género? El drama es el más frecuente en ambos (32,0 %
entre las mujeres y personas no binarias, 32,8 % entre los hombres), así que
no las distingue. La mayor diferencia a su favor es el documental
(27,0 % frente a 18,3 %, +8,7 puntos). Los hombres pesan más en comedia,
acción, terror y suspense.

In [7]:
comp = leer_resultado("generos_por_grupo.csv").set_index("genero")
comp = comp.drop(index="Sin género", errors="ignore").sort_values("dif_pp")
# dif_pp = peso del género entre mujeres y personas no binarias menos peso entre hombres
display(comp.head(5))

,Mujeres y personas no binarias,Hombres,dif_pp
genero,,,
Comedia,13.37,20.16,-6.80
Acción,1.06,6.97,-5.91
Terror,4.34,9.51,-5.17
Suspense,4.39,9.40,-5.01
Crimen,1.88,5.30,-3.42


Arriba, los cinco géneros con más peso entre los hombres (diferencia negativa).
Los dos únicos con más peso entre las mujeres y personas no binarias son:

In [8]:
display(comp[comp["dif_pp"] > 0].sort_values("dif_pp", ascending=False))

,Mujeres y personas no binarias,Hombres,dif_pp
genero,,,
Documental,27.02,18.35,8.67
Animación,7.26,5.23,2.03


### 6.5 Evolución en el tiempo

![Evolución del porcentaje de mujeres](../figuras/08_evolucion_mujeres.png)

El porcentaje sube en los tres niveles: con todas las películas, del 13,3 %
(2000) al 24,7 % (2025). La curva de las películas con muchos votos es más
irregular porque hay pocas películas por año.

![Evolución por género y periodo](../figuras/09_evolucion_por_genero_periodo.png)

In [9]:
per = leer_resultado("evolucion_por_genero_periodo.csv").set_index("Unnamed: 0")
per.index.name = "género"
display(per)

,2000-04,2005-09,2010-14,2015-19,2020-25,cambio_pp
género,,,,,,
Drama,13.3,14.2,17.3,21.5,23.8,10.5
Documental,19.4,21.9,23.8,28.2,30.3,10.9
Comedia,9.5,10.3,11.9,15.3,18.2,8.7
Terror,3.2,5.1,6.9,10.5,14.4,11.2


Sube en los cuatro géneros analizados: documental del 19,4 % al 30,3 %,
drama del 13,3 % al 23,8 %, comedia del 9,5 % al 18,2 % y terror del 3,2 %
al 14,4 %. El orden entre géneros se mantiene en todos los periodos. El
último periodo abarca seis años y los demás cinco.

### 6.6 ¿Crece solo porque hay más películas?

![Mujeres frente a hombres](../figuras/11_mujeres_frente_a_hombres.png)

El porcentaje ya tiene en cuenta el crecimiento del catálogo, pero comparo
también los números absolutos. Entre 2000-2004 y 2021-2025, las películas
dirigidas por mujeres y personas no binarias pasan de 3.708 a 21.363
(×5,8) y las dirigidas por hombres, de 23.472 a 65.356 (×2,8). Por cada 100
películas de hombres había 15,8 de mujeres y personas no binarias en
2000-2004 y hay 32,7 en 2021-2025. El aumento no se explica solo porque se
registren más películas: las de mujeres y personas no binarias crecen más
rápido. La comparación solo incluye películas con el género de la dirección
conocido, así que sigue afectada por la cautela del apartado siguiente.

### 6.7 La gran cautela: la dirección «no identificada»

![Dirección no identificada por año](../figuras/10_no_identificado_por_anio.png)

El porcentaje de películas con la dirección no identificada pasa del 35 %
en 2000 al 62 % en 2025. La base con dirección conocida es, por tanto,
proporcionalmente menor en los años recientes.

In [10]:
ni = leer_resultado("no_identificado_por_anio.csv").set_index("anio_estreno")["pct_no_identificado"]
print(f"2000: {ni.loc[2000]} %   2010: {ni.loc[2010]} %   2020: {ni.loc[2020]} %   2025: {ni.loc[2025]} %")

2000: 34.99 %   2010: 45.46 %   2020: 58.13 %   2025: 61.66 %


## 7. Conclusiones

**Hipótesis 1 — Más hombres que mujeres en todos los géneros: se confirma.**
Ningún género supera el 29,6 % de películas con alguna mujer en la dirección
(documental, todas las películas), así que los hombres dirigen al menos el
70 % en todos. La hipótesis revisada (menos del 50 % en todos los géneros)
también se cumple, aunque era un resultado esperable con un 20 % global.

**Hipótesis 2 — Aumento en 25 años: se confirma.** Entre 2000-04 y 2021-25 el
porcentaje pasa del 13,5 % al 24,2 %, con una subida de 0,47 a 0,54 puntos
por año según el nivel de visibilidad. Aparece en todos los niveles y en
los cuatro géneros analizados. Frente a los hombres, pasa de 15,8 a 32,7
películas por cada 100 dirigidas por hombres (×5,8 frente a ×2,8), por lo que
no se debe solo a que haya más películas.

**Hipótesis 3 — El drama como género principal de las mujeres: solo en
parte.** Es el más frecuente en su filmografía (32,0 %), pero también en la
de los hombres (32,8 %). Lo que las distingue es el documental (+8,7 puntos).
El género con mayor presencia de mujeres es el documental (26,7 %) y el de
menor, la acción (3,7 %).

**Otros resultados.** El porcentaje global baja del 20,2 % (todas las
películas) al 16,6 % (≥ 1 voto) y al 10,7 % (≥ 100 votos): las películas
dirigidas por mujeres y personas no binarias están menos representadas
entre las más votadas.

### Limitaciones

- **Género de la dirección no identificado en el 53,6 % de la base**, y
  creciente con los años (35 % → 62 %). El aumento observado puede estar
  afectado y no se puede determinar en qué sentido con estos datos. Que
  la subida aparezca en todos los niveles y géneros apoya la tendencia, pero
  no la confirma de forma definitiva.
- **Descarte no aleatorio** de películas sin director acreditado.
- **El género procede de TMDB**, no de la autoidentificación de las personas.
- **El catálogo de TMDB no es todo el cine**: lo edita la comunidad y crece
  más en los años recientes. El estudio describe las películas con dirección
  registrada en TMDB.
- **La hipótesis revisada se formuló después de ver los datos.**

### Próximos pasos

- Analizar la presencia de películas dirigidas por mujeres en listas de
  mejores películas.
- Completar el género de la dirección con otra fuente para reducir el
  grupo «No identificado».